In [1]:
import subprocess

subprocess.run([
    "pip", "install",
    "azure-storage-blob",
    "pyarrow",
    "pandas",
    "-q"
])

CompletedProcess(args=['pip', 'install', 'azure-storage-blob', 'pyarrow', 'pandas', '-q'], returncode=0)

In [2]:
import os
import sys
from pyspark.sql import SparkSession

ACCOUNT_NAME = os.environ["AZURE_STORAGE_ACCOUNT_NAME"]
ACCOUNT_KEY  = os.environ["AZURE_STORAGE_ACCOUNT_KEY"]
CONTAINER    = os.environ.get("AZURE_CONTAINER_NAME", "nyc-traffic-lakehouse")

spark = (SparkSession.builder
    .appName("Train_RandomForest_NYC_Traffic")
    .master("spark://spark-master:7077")

    # Timezone phải thống nhất với 03 và 04
    .config("spark.sql.session.timeZone", "America/New_York")

    # Tránh lỗi Python mismatch giữa Jupyter và Spark worker
    .config("spark.pyspark.python", "/usr/bin/python3")
    .config("spark.executorEnv.PYSPARK_PYTHON", "/usr/bin/python3")

    # Azure + Delta
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-azure:3.3.4,"
            "io.delta:delta-spark_2.13:4.0.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config(f"spark.hadoop.fs.azure.account.key.{ACCOUNT_NAME}.blob.core.windows.net", ACCOUNT_KEY)

    # Tối ưu cho dataset lớn
    .config("spark.sql.shuffle.partitions", "128")
    .config("spark.default.parallelism", "128")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")

    .getOrCreate())

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)
print("Driver Python:", sys.executable)

Spark version: 4.0.0
Master: spark://spark-master:7077
Driver Python: /opt/conda/bin/python


In [3]:
BASE_PATH = f"wasbs://{CONTAINER}@{ACCOUNT_NAME}.blob.core.windows.net"

GOLD_PATH  = f"{BASE_PATH}/gold/training_features"
MODEL_PATH = f"{BASE_PATH}/artifacts/model/random_forest_model"

print("GOLD_PATH :", GOLD_PATH)
print("MODEL_PATH:", MODEL_PATH)

GOLD_PATH : wasbs://nyc-traffic-lakehouse@nyctrafficlakehouse.blob.core.windows.net/gold/training_features
MODEL_PATH: wasbs://nyc-traffic-lakehouse@nyctrafficlakehouse.blob.core.windows.net/artifacts/model/random_forest_model


In [4]:
df = spark.read.format("delta").load(GOLD_PATH)

print(f"Tổng rows Gold: {df.count():,}")
print(f"Số link_id unique: {df.select('link_id').distinct().count()}")

df.printSchema()
df.show(5, truncate=False)

Tổng rows Gold: 21,887,219
Số link_id unique: 116
root
 |-- link_id: string (nullable = true)
 |-- borough: string (nullable = true)
 |-- hour: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- is_weekend: integer (nullable = true)
 |-- is_holiday: integer (nullable = true)
 |-- speed_ratio: double (nullable = true)
 |-- current_congestion: integer (nullable = true)
 |-- past_congestion_15min: integer (nullable = true)
 |-- past_speed_ratio_15min: double (nullable = true)
 |-- past_congestion_30min: integer (nullable = true)
 |-- past_speed_ratio_30min: double (nullable = true)
 |-- past_congestion_45min: integer (nullable = true)
 |-- past_speed_ratio_45min: double (nullable = true)
 |-- past_congestion_60min: integer (nullable = true)
 |-- past_speed_ratio_60min: double (nullable = true)
 |-- congestion_trend: integer (nullable = true)
 |-- speed_ratio_trend: double (nullable = true)
 |-- future_congestion_15min: integer 

In [5]:
from pyspark.sql import functions as F

label_col = "future_congestion_15min"

feature_cols = [
    "link_id", "borough",
    "hour", "day_of_week", "month", "is_weekend", "is_holiday",
    "speed_ratio", "current_congestion",
    "past_congestion_15min", "past_speed_ratio_15min",
    "past_congestion_30min", "past_speed_ratio_30min",
    "past_congestion_45min", "past_speed_ratio_45min",
    "past_congestion_60min", "past_speed_ratio_60min",
    "congestion_trend", "speed_ratio_trend"
]

assert len(feature_cols) == 19, f"Sai số lượng feature, hiện có {len(feature_cols)}"

required_cols = feature_cols + [label_col, "data_as_of", "year"]

missing = [c for c in required_cols if c not in df.columns]
assert len(missing) == 0, f"Thiếu cột: {missing}"

df_model = df.select(*required_cols)

# Ép kiểu an toàn
df_model = (df_model
    .withColumn("link_id", F.col("link_id").cast("string"))
    .withColumn("borough", F.col("borough").cast("string"))
    .withColumn(label_col, F.col(label_col).cast("double"))
)

# Drop null lần cuối trước train
df_model = df_model.dropna(subset=feature_cols + [label_col])

print(f"Rows dùng cho model: {df_model.count():,}")
df_model.groupBy(label_col).count().orderBy(label_col).show()

Rows dùng cho model: 21,887,219
+-----------------------+--------+
|future_congestion_15min|   count|
+-----------------------+--------+
|                    0.0|15000112|
|                    1.0| 3227613|
|                    2.0| 3659494|
+-----------------------+--------+



In [6]:
df_train = df_model.filter(
    (F.col("year") == 2024) |
    ((F.col("year") == 2025) & (F.col("month") <= 9))
)

df_val = df_model.filter(
    (F.col("year") == 2025) &
    (F.col("month").between(10, 12))
)

df_test = df_model.filter(
    F.col("year") >= 2026
)

print(f"Train rows: {df_train.count():,}")
print(f"Val rows  : {df_val.count():,}")
print(f"Test rows : {df_test.count():,}")

print("Train label distribution:")
df_train.groupBy(label_col).count().orderBy(label_col).show()

print("Val label distribution:")
df_val.groupBy(label_col).count().orderBy(label_col).show()

print("Test label distribution:")
df_test.groupBy(label_col).count().orderBy(label_col).show()

Train rows: 15,791,119
Val rows  : 2,271,083
Test rows : 3,820,505
Train label distribution:
+-----------------------+--------+
|future_congestion_15min|   count|
+-----------------------+--------+
|                    0.0|10920118|
|                    1.0| 2317579|
|                    2.0| 2553422|
+-----------------------+--------+

Val label distribution:
+-----------------------+-------+
|future_congestion_15min|  count|
+-----------------------+-------+
|                    0.0|1552836|
|                    1.0| 328685|
|                    2.0| 389562|
+-----------------------+-------+

Test label distribution:
+-----------------------+-------+
|future_congestion_15min|  count|
+-----------------------+-------+
|                    0.0|2523587|
|                    1.0| 581041|
|                    2.0| 715877|
+-----------------------+-------+



In [7]:
class_counts = df_train.groupBy(label_col).count()

total_train = df_train.count()
num_classes = 3

weights_df = class_counts.withColumn(
    "class_weight",
    F.lit(total_train) / (F.lit(num_classes) * F.col("count"))
).select(
    F.col(label_col).alias("label_for_weight"),
    "class_weight"
)

weights_df.orderBy("label_for_weight").show()

df_train_w = df_train.join(
    F.broadcast(weights_df),
    df_train[label_col] == weights_df["label_for_weight"],
    "left"
).drop("label_for_weight")

df_train_w = df_train_w.withColumn(
    "class_weight",
    F.col("class_weight").cast("double")
)

df_train_w.select(label_col, "class_weight").show(10)

+----------------+------------------+
|label_for_weight|      class_weight|
+----------------+------------------+
|             0.0|0.4820191808672153|
|             1.0|2.2712090217133194|
|             2.0|2.0614322009183494|
+----------------+------------------+

+-----------------------+------------------+
|future_congestion_15min|      class_weight|
+-----------------------+------------------+
|                    1.0|2.2712090217133194|
|                    0.0|0.4820191808672153|
|                    2.0|2.0614322009183494|
|                    0.0|0.4820191808672153|
|                    2.0|2.0614322009183494|
|                    0.0|0.4820191808672153|
|                    2.0|2.0614322009183494|
|                    0.0|0.4820191808672153|
|                    2.0|2.0614322009183494|
|                    0.0|0.4820191808672153|
+-----------------------+------------------+
only showing top 10 rows


In [8]:
# Tỷ lệ sample cho từng class.
# Có thể chỉnh:
# - Máy yếu: 0.05
# - Máy ổn: 0.10
# - Muốn kỹ hơn: 0.15 hoặc 0.20
TUNE_SAMPLE_FRACTION = 0.03

fractions = {
    0.0: TUNE_SAMPLE_FRACTION,
    1.0: TUNE_SAMPLE_FRACTION,
    2.0: TUNE_SAMPLE_FRACTION,
}

df_train_tune = df_train_w.sampleBy(
    label_col,
    fractions=fractions,
    seed=42
)

df_val_tune = df_val.sampleBy(
    label_col,
    fractions=fractions,
    seed=42
)

# Chỉ giữ cột cần thiết để giảm RAM/shuffle
train_keep_cols = feature_cols + [label_col, "class_weight"]
val_keep_cols = feature_cols + [label_col]

df_train_tune = df_train_tune.select(*train_keep_cols).repartition(128, "link_id").cache()
df_val_tune = df_val_tune.select(*val_keep_cols).repartition(128, "link_id").cache()

print(f"Train tune rows: {df_train_tune.count():,}")
print(f"Val tune rows  : {df_val_tune.count():,}")

print("Train tune distribution:")
df_train_tune.groupBy(label_col).count().orderBy(label_col).show()

print("Val tune distribution:")
df_val_tune.groupBy(label_col).count().orderBy(label_col).show()

Train tune rows: 472,638
Val tune rows  : 67,956
Train tune distribution:
+-----------------------+------+
|future_congestion_15min| count|
+-----------------------+------+
|                    0.0|326319|
|                    1.0| 69809|
|                    2.0| 76510|
+-----------------------+------+

Val tune distribution:
+-----------------------+-----+
|future_congestion_15min|count|
+-----------------------+-----+
|                    0.0|46503|
|                    1.0| 9765|
|                    2.0|11688|
+-----------------------+-----+



In [9]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier

categorical_indexers = [
    StringIndexer(
        inputCol="link_id",
        outputCol="link_id_idx",
        handleInvalid="keep"
    ),
    StringIndexer(
        inputCol="borough",
        outputCol="borough_idx",
        handleInvalid="keep"
    )
]

numeric_features = [
    "hour", "day_of_week", "month", "is_weekend", "is_holiday",
    "speed_ratio", "current_congestion",
    "past_congestion_15min", "past_speed_ratio_15min",
    "past_congestion_30min", "past_speed_ratio_30min",
    "past_congestion_45min", "past_speed_ratio_45min",
    "past_congestion_60min", "past_speed_ratio_60min",
    "congestion_trend", "speed_ratio_trend"
]

assembler_inputs = [
    "link_id_idx",
    "borough_idx",
] + numeric_features

assembler = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol="features",
    handleInvalid="keep"
)

def build_pipeline(numTrees, maxDepth, minInstancesPerNode):
    rf = RandomForestClassifier(
        featuresCol="features",
        labelCol=label_col,
        weightCol="class_weight",
        predictionCol="prediction",
        probabilityCol="probability",
        rawPredictionCol="rawPrediction",

        numTrees=numTrees,
        maxDepth=maxDepth,
        minInstancesPerNode=minInstancesPerNode,

        maxBins=132,
        impurity="gini",
        featureSubsetStrategy="sqrt",
        seed=42,

        # Tối ưu tốc độ / tránh quá nặng
        cacheNodeIds=False,
        subsamplingRate=0.8
    )

    return Pipeline(stages=categorical_indexers + [assembler, rf])

In [10]:
def compute_macro_f1(pred_df, label_col="future_congestion_15min", prediction_col="prediction"):
    """
    Tính Macro F1 thủ công cho 3 class: 0, 1, 2.
    Return: macro_f1, per_class_metrics_df
    """

    classes = [0.0, 1.0, 2.0]

    metrics = []

    for c in classes:
        tp = pred_df.filter(
            (F.col(label_col) == c) & (F.col(prediction_col) == c)
        ).count()

        fp = pred_df.filter(
            (F.col(label_col) != c) & (F.col(prediction_col) == c)
        ).count()

        fn = pred_df.filter(
            (F.col(label_col) == c) & (F.col(prediction_col) != c)
        ).count()

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

        metrics.append((c, tp, fp, fn, precision, recall, f1))

    macro_f1 = sum(row[-1] for row in metrics) / len(classes)

    metrics_df = spark.createDataFrame(
        metrics,
        ["class", "tp", "fp", "fn", "precision", "recall", "f1"]
    )

    return macro_f1, metrics_df

In [11]:
import time
import pandas as pd

COMBOS = [
    {"numTrees": 50, "maxDepth": 8,  "minInstancesPerNode": 5},
    {"numTrees": 50, "maxDepth": 10, "minInstancesPerNode": 5},
    {"numTrees": 50, "maxDepth": 12, "minInstancesPerNode": 5},
    {"numTrees": 100, "maxDepth": 10, "minInstancesPerNode": 5},
#    {"numTrees": 200, "maxDepth": 15, "minInstancesPerNode": 1},
#    {"numTrees": 200, "maxDepth": 15, "minInstancesPerNode": 2},
 #   {"numTrees": 200, "maxDepth": 20, "minInstancesPerNode": 1},
  #  {"numTrees": 200, "maxDepth": 20, "minInstancesPerNode": 2},
   # {"numTrees": 200, "maxDepth": 25, "minInstancesPerNode": 1},
   # {"numTrees": 200, "maxDepth": 25, "minInstancesPerNode": 2},
]

results = []
best_macro_f1 = -1
best_params = None
best_model_tune = None

for i, params in enumerate(COMBOS, start=1):
    print("\n" + "=" * 80)
    print(f"[{i}/{len(COMBOS)}] Training params: {params}")
    print("=" * 80)

    t0 = time.time()

    pipeline = build_pipeline(
        numTrees=params["numTrees"],
        maxDepth=params["maxDepth"],
        minInstancesPerNode=params["minInstancesPerNode"]
    )

    model = pipeline.fit(df_train_tune)

    fit_time = time.time() - t0
    print(f"Fit xong sau {fit_time/60:.2f} phút")

    t1 = time.time()

    pred_val = model.transform(df_val_tune).select(label_col, "prediction").cache()
    pred_count = pred_val.count()

    macro_f1, metrics_df = compute_macro_f1(pred_val, label_col, "prediction")

    eval_time = time.time() - t1

    print(f"Val rows predict: {pred_count:,}")
    print(f"Macro F1: {macro_f1:.6f}")
    print(f"Eval time: {eval_time/60:.2f} phút")
    metrics_df.orderBy("class").show()

    result = {
        **params,
        "macro_f1": macro_f1,
        "fit_time_min": fit_time / 60,
        "eval_time_min": eval_time / 60,
        "val_rows": pred_count
    }
    results.append(result)

    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        best_params = params
        best_model_tune = model

    pred_val.unpersist()

results_pdf = pd.DataFrame(results).sort_values("macro_f1", ascending=False)
results_pdf


[1/4] Training params: {'numTrees': 50, 'maxDepth': 8, 'minInstancesPerNode': 5}
Fit xong sau 1.30 phút
Val rows predict: 67,956
Macro F1: 0.758265
Eval time: 0.44 phút
+-----+-----+----+----+------------------+------------------+------------------+
|class|   tp|  fp|  fn|         precision|            recall|                f1|
+-----+-----+----+----+------------------+------------------+------------------+
|  0.0|40260|1811|6243|0.9569537210905374|0.8657505967356944|0.9090703818276245|
|  1.0| 6793|7033|2972|0.4913207001301895|0.6956477214541731|0.5758975880632444|
|  2.0| 9378|2681|2310|  0.77767642424745|0.8023613963039015|0.7898260832947319|
+-----+-----+----+----+------------------+------------------+------------------+


[2/4] Training params: {'numTrees': 50, 'maxDepth': 10, 'minInstancesPerNode': 5}


Py4JJavaError: An error occurred while calling o1822.fit.
: org.apache.spark.SparkException: Job aborted due to stage failure: ShuffleMapStage 279 (mapPartitions at RandomForest.scala:646) has failed the maximum allowable number of times: 4. Most recent failure reason:
org.apache.spark.shuffle.MetadataFetchFailedException: Missing an output location for shuffle 17 partition 34
	at org.apache.spark.MapOutputTracker$.validateStatus(MapOutputTracker.scala:1770)
	at org.apache.spark.MapOutputTracker$.$anonfun$convertMapStatuses$11(MapOutputTracker.scala:1715)
	at org.apache.spark.MapOutputTracker$.$anonfun$convertMapStatuses$11$adapted(MapOutputTracker.scala:1714)
	at scala.collection.IterableOnceOps.foreach(IterableOnce.scala:619)
	at scala.collection.IterableOnceOps.foreach$(IterableOnce.scala:617)
	at scala.collection.AbstractIterator.foreach(Iterator.scala:1306)
	at org.apache.spark.MapOutputTracker$.convertMapStatuses(MapOutputTracker.scala:1714)
	at org.apache.spark.MapOutputTrackerW

In [ ]:
print("BEST PARAMS:", best_params)
print("BEST MACRO F1:", best_macro_f1)

In [ ]:
df_train_val = df_model.filter(
    (F.col("year") == 2024) |
    (F.col("year") == 2025)
)

print(f"Train+Val rows: {df_train_val.count():,}")
df_train_val.groupBy(label_col).count().orderBy(label_col).show()

In [ ]:
class_counts_final = df_train_val.groupBy(label_col).count()
total_train_val = df_train_val.count()

weights_final_df = class_counts_final.withColumn(
    "class_weight",
    F.lit(total_train_val) / (F.lit(num_classes) * F.col("count"))
).select(
    F.col(label_col).alias("label_for_weight"),
    "class_weight"
)

weights_final_df.orderBy("label_for_weight").show()

df_train_val_w = df_train_val.join(
    F.broadcast(weights_final_df),
    df_train_val[label_col] == weights_final_df["label_for_weight"],
    "left"
).drop("label_for_weight")

df_train_val_w = df_train_val_w.select(
    *(feature_cols + [label_col, "class_weight"])
).repartition(128, "link_id").cache()

print(f"Train+Val weighted rows: {df_train_val_w.count():,}")

In [ ]:
print("Best params dùng để refit full:", best_params)

final_pipeline = build_pipeline(
    numTrees=best_params["numTrees"],
    maxDepth=best_params["maxDepth"],
    minInstancesPerNode=best_params["minInstancesPerNode"]
)

t0 = time.time()

final_model = final_pipeline.fit(df_train_val_w)

print(f"Refit final model xong sau {(time.time() - t0)/60:.2f} phút")

In [ ]:
final_model.write().overwrite().save(MODEL_PATH)

print(f"Đã lưu model tại: {MODEL_PATH}")

In [ ]:
from pyspark.ml import PipelineModel

loaded_model = PipelineModel.load(MODEL_PATH)

print("Load model thành công")
print(loaded_model)